# Limpieza de mediciones de PM10 y PM2.5

La preparación utiliza los hallazgos del análisis exploratorio de Evaluaciones Ambientales de Seguimiento — Aire (OEFA). Su objetivo es obtener **mediciones individuales normalizadas y trazables**, con identidad espacial y temporal suficiente para preparar posteriormente observaciones de PM10 y PM2.5.

Se retiran parámetros ajenos al estudio, resultados no cuantificables, unidades incompatibles, registros sin identidad esencial, zonas UTM no admitidas, inconsistencias temporales comprobables, duplicados en la clave definida y claves ambiguas. No se imputan concentraciones, no se promedian intervalos distintos ni se eliminan extremos solo por su magnitud. Los ceros se conservan como valores registrados, sin interpretarlos automáticamente como ausencia de contaminación.

## 1. Decisiones de limpieza sustentadas en el análisis

| Hallazgo | Tratamiento |
| --- | --- |
| Gases, meteorología y otros parámetros | Seleccionar exclusivamente PM10 y PM2.5. |
| Códigos como `-`, `NA`, `CA`, `<LC` y valores no finitos | Excluir cuando no representan una concentración numérica directamente utilizable; conservar el motivo en la auditoría. |
| Variantes tipográficas de µg/m³, incluyendo `ugm3` | Normalizar mediante equivalencias explícitas; no convertir ppb. |
| Fechas seriales y horas como fracción de día | Interpretar con la misma convención del EDA y conservar las representaciones originales. |
| `24 Horas`, `24 horas` y espacios redundantes | Homogeneizar etiquetas; mantener separadas las mediciones de una hora y de 24 horas. |
| Zonas como `3600` o `4285` | Excluir del conjunto preparado; no reemplazarlas por una zona supuesta. |
| Variantes `WGS84`, `WGS 84`, `WGS-84` | Homogeneizar a `WGS84`. Otros valores, por ejemplo `WSG84`, permanecen identificados y señalados para revisión. |
| Final de medición ausente | Conservar como desconocido; no exigirlo para todas las mediciones horarias ni inventar un final. |
| Final interpretable anterior al inicio | Separar para revisión y excluir del conjunto preparado. |
| Extremos IQR/P99 y concentraciones cero | Conservar: una señal estadística no demuestra un error de medición. |
| Repeticiones tras normalizar | Retener una representación por clave de identidad y resultado, con un balance de duplicados. |
| Varias mediciones por contaminante en la misma clave de observación | Separar todas las involucradas; no resolverlas mediante promedio o selección arbitraria. |

La validación de zona y rangos de coordenadas es un control estructural, no una certificación cartográfica. Un datum pendiente de revisión no se utiliza para inferir distancias geográficas; permanece como parte de la identidad del registro.

In [1]:
%pip -q install duckdb

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Entrada y configuración

Se procesa el archivo original para que la limpieza sea reproducible en una sesión independiente del EDA. Solo se materializan las mediciones de los dos contaminantes seleccionados. Cada ejecución utiliza una carpeta nueva y no modifica el CSV de origen.

In [3]:
from pathlib import Path
from datetime import datetime, timezone
import json,os,platform,shutil,time
import duckdb
import pandas as pd
from IPython.display import display,Markdown

DATA_DIR=Path('/content/drive/MyDrive/Concurrente')
RAW_CSV=DATA_DIR/'EAS_Aire_0.csv'
if not RAW_CSV.is_file():
    raise FileNotFoundError('No se encontró EAS_Aire_0.csv. Revisa DATA_DIR.')
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUT=DATA_DIR/'Limpieza_PM'/RUN_ID
OUT.mkdir(parents=True,exist_ok=False)
LOCAL=Path('/content')/('limpieza_pm_'+RUN_ID)
LOCAL.mkdir(parents=True,exist_ok=False)
if shutil.disk_usage('/content').free<10*1024**3:
    raise RuntimeError('Se requieren al menos 10 GiB libres para la base y temporales.')
con=duckdb.connect(str(LOCAL/'limpieza.duckdb'))
con.execute("SET memory_limit='3GB'")
con.execute(f'SET threads={min(os.cpu_count() or 2,4)}')
con.execute('SET preserve_insertion_order=false')
def sqlpath(p):return str(p).replace("'","''")
con.execute("SET temp_directory='"+sqlpath(LOCAL/'spill')+"'")
tables={}
def report(name,sql):
    frame=con.execute(sql).df()
    tables[name]=frame
    frame.to_csv(OUT/(name+'.csv'),index=False)
    display(Markdown('**'+name+'**'))
    display(frame.head(50))
    if len(frame)>50:print(f'Se muestran 50 de {len(frame):,} filas. El CSV conserva la tabla completa.')
    return frame
provenance={'run_id':RUN_ID,'etapa':'limpieza','fecha_utc':datetime.now(timezone.utc).isoformat(),
 'archivo_fuente':RAW_CSV.name,'bytes_fuente':RAW_CSV.stat().st_size,
 'python':platform.python_version(),'duckdb':duckdb.__version__,
 'cobertura':'todos los años disponibles','estado':'iniciado'}
(OUT/'ejecucion.json').write_text(json.dumps(provenance,indent=2,ensure_ascii=False),encoding='utf-8')
display(pd.DataFrame([{'archivo':RAW_CSV.name,'bytes':RAW_CSV.stat().st_size,
 'componentes':'PM10 y PM2.5','restriccion_de_anios':'ninguna'}]))

,archivo,bytes,componentes,restriccion_de_anios
0,EAS_Aire_0.csv,5153626505,PM10 y PM2.5,ninguna


## 3. Selección de variables y conversión

Se requieren columnas de procedencia, punto de monitoreo, coordenadas y referencia espacial, inicio y final de medición, periodo, parámetro, valor y unidad. Las columnas administrativas, descriptivas o de otros componentes no se incluyen en la salida de modelado; su origen permanece disponible en el archivo original.

Los valores temporales originales se mantienen para auditar conversiones. Las fracciones horarias entre 0 y 1 se interpretan como partes del día, con precisión de segundos. No se aplica un límite de antigüedad.

In [4]:
con.execute(f"""CREATE VIEW raw AS SELECT * FROM read_csv('{sqlpath(RAW_CSV)}',
 header=true,auto_detect=true,all_varchar=true,sample_size=100000,ignore_errors=false)""")
required=['Número.de.informe','Nombre.de.la.Evaluación','Nombre.del.punto','Este','Norte',
 'Zona','Datum','Altitud','Periodo','Fecha.inicio','Hora.inicio','Fecha.fin','Hora.fin',
 'Parámetro','Valor','Unidad.de.medida']
available=set(con.execute('DESCRIBE raw').df().column_name)
if set(required)-available:
    raise ValueError('Faltan columnas: '+str(sorted(set(required)-available)))
n_source=int(con.execute('SELECT count(*) FROM raw').fetchone()[0])
con.execute("""CREATE MACRO fecha_interpretada(x) AS coalesce(
 try_cast(trim(x) AS DATE),
 cast(try_strptime(trim(x), ['%d/%m/%Y','%d-%m-%Y','%Y/%m/%d']) AS DATE),
 CASE WHEN isfinite(try_cast(x AS DOUBLE)) AND try_cast(x AS DOUBLE) BETWEEN 0 AND 2958465
 THEN DATE '1899-12-30' + try_cast(floor(try_cast(x AS DOUBLE)) AS INTEGER) END)""")
con.execute("""CREATE MACRO hora_interpretada(x) AS coalesce(
 try_cast(nullif(trim(x),'-') AS TIME),
 CASE WHEN isfinite(try_cast(replace(trim(x),',','.') AS DOUBLE))
 AND try_cast(replace(trim(x),',','.') AS DOUBLE)>=0
 AND try_cast(replace(trim(x),',','.') AS DOUBLE)<1
 THEN cast(TIMESTAMP '2000-01-01' + cast(least(86399,
 round(try_cast(replace(trim(x),',','.') AS DOUBLE)*86400)) AS BIGINT)*INTERVAL 1 SECOND AS TIME) END)""")

tic=time.perf_counter()
con.execute("""CREATE TABLE seleccionadas AS WITH base AS (
 SELECT trim("Número.de.informe") AS informe,trim("Nombre.de.la.Evaluación") AS evaluacion,
 trim("Nombre.del.punto") AS punto,trim("Este") AS este_original,trim("Norte") AS norte_original,
 trim("Zona") AS zona_original,trim("Datum") AS datum_original,
 trim("Altitud") AS altitud_original,
 try_cast("Este" AS DOUBLE) AS este,try_cast("Norte" AS DOUBLE) AS norte,
 try_cast("Altitud" AS DOUBLE) AS altitud,
 CASE WHEN isfinite(try_cast("Zona" AS DOUBLE))
 AND try_cast("Zona" AS DOUBLE)=trunc(try_cast("Zona" AS DOUBLE))
 THEN try_cast("Zona" AS INTEGER) END AS zona,
 CASE WHEN upper(regexp_replace(trim("Datum"),'[ -]','','g'))='WGS84' THEN 'WGS84'
 ELSE trim("Datum") END AS datum,
 trim("Periodo") AS periodo_original,
 CASE lower(regexp_replace(trim("Periodo"),'\\s+',' ','g'))
 WHEN '1 hora' THEN '1 hora' WHEN '24 horas' THEN '24 horas' END AS periodo,
 trim("Fecha.inicio") AS fecha_original,fecha_interpretada("Fecha.inicio") AS fecha,
 trim("Hora.inicio") AS hora_original,hora_interpretada("Hora.inicio") AS hora,
 trim("Fecha.fin") AS fecha_fin_original,fecha_interpretada("Fecha.fin") AS fecha_fin,
 trim("Hora.fin") AS hora_fin_original,hora_interpretada("Hora.fin") AS hora_fin,
 trim("Parámetro") AS parametro_original,trim("Valor") AS valor_original,
 trim("Unidad.de.medida") AS unidad_original,
 lower(replace(replace(replace(trim("Unidad.de.medida"),'μ','µ'),'³','3'),' ','')) AS unidad_norm,
 try_cast(replace(trim("Valor"),',','.') AS DOUBLE) AS concentracion,
 CASE WHEN regexp_matches(lower("Parámetro"),'pm *2[.,]5|part[ií]cul.*2[.,]5') THEN 'PM2.5'
 WHEN regexp_matches(lower("Parámetro"),'pm *10|part[ií]cul.*10') THEN 'PM10' END AS contaminante
 FROM raw)
 SELECT *,CASE WHEN unidad_norm IN ('µg/m3','ug/m3','µgm3','ugm3') THEN 'µg/m³' END AS unidad,
 fecha+hora AS inicio,fecha_fin+hora_fin AS fin,
 coalesce(cast(fecha_fin AS VARCHAR),fecha_fin_original,'') AS fecha_fin_clave,
 coalesce(cast(hora_fin AS VARCHAR),hora_fin_original,'') AS hora_fin_clave
 FROM base WHERE contaminante IS NOT NULL""")
n_pm=int(con.execute('SELECT count(*) FROM seleccionadas').fetchone()[0])
lectura_segundos=time.perf_counter()-tic
report('01_entrada',"""SELECT contaminante,count(*) AS mediciones,
 min(fecha) AS primera_fecha,max(fecha) AS ultima_fecha FROM seleccionadas GROUP BY contaminante ORDER BY contaminante""")
print('Filas de la fuente:',f'{n_source:,}','| Mediciones PM seleccionadas:',f'{n_pm:,}')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

**01_entrada**

,contaminante,mediciones,primera_fecha,ultima_fecha
0,PM10,894366,2018-01-01,2024-12-31
1,PM2.5,686264,2019-01-25,2024-12-31


Filas de la fuente: 11,407,394 | Mediciones PM seleccionadas: 1,580,630


## 4. Exclusiones y balance secuencial

Cada medición se clasifica por el **primer criterio incumplido**. Los motivos son excluyentes y permiten reconciliar entrada, descartes y salida. Se exige hora de inicio y periodo identificable porque forman parte de la observación posterior.

Se admiten zonas UTM 17, 18 y 19 y rangos estructurales de coordenadas. Los finales desconocidos permanecen nulos; solo se excluye la contradicción cuando inicio y final completos permiten comprobarla. No se completa una fecha u hora faltante utilizando valores de otra medición.

In [5]:
rules=[
 ('Concentración no cuantificable',"concentracion IS NULL OR NOT isfinite(concentracion)"),
 ('Concentración negativa',"concentracion<0"),
 ('Unidad no compatible',"unidad IS NULL"),
 ('Procedencia incompleta',"informe IS NULL OR upper(informe) IN ('','-','NA','N.I.') OR evaluacion IS NULL OR upper(evaluacion) IN ('','-','NA','N.I.')"),
 ('Punto no informado',"punto IS NULL OR upper(punto) IN ('','-','NA','N.I.')"),
 ('Inicio no interpretable',"fecha IS NULL OR hora IS NULL"),
 ('Periodo no reconocido',"periodo IS NULL"),
 ('Zona UTM no admitida',"zona IS NULL OR zona NOT IN (17,18,19)"),
 ('Coordenadas fuera del control estructural',"este IS NULL OR norte IS NULL OR NOT isfinite(este) OR NOT isfinite(norte) OR este<100000 OR este>900000 OR norte<=0 OR norte>10000000"),
 ('Datum no informado',"datum IS NULL OR upper(datum) IN ('','-','NA','N.I.')"),
 ('Final anterior al inicio',"fin IS NOT NULL AND fin<inicio")]
cases=' '.join("WHEN "+condition+" THEN '"+label.replace("'","''")+"'" for label,condition in rules)
con.execute("CREATE TABLE clasificadas AS SELECT *,CASE "+cases+" ELSE 'Conservar' END AS motivo FROM seleccionadas")
exclusions=report('02_exclusiones_por_motivo',"""SELECT motivo,contaminante,count(*) AS registros
 FROM clasificadas WHERE motivo<>'Conservar' GROUP BY ALL ORDER BY motivo,contaminante""")
rows=[{'etapa':'Fuente completa','filas':n_source,'descartes_en_etapa':0},
      {'etapa':'PM10 y PM2.5','filas':n_pm,'descartes_en_etapa':n_source-n_pm}]
remaining=n_pm
for label,_ in rules:
    removed=int(con.execute('SELECT count(*) FROM clasificadas WHERE motivo=?',[label]).fetchone()[0])
    remaining-=removed
    rows.append({'etapa':label,'filas':remaining,'descartes_en_etapa':removed})
con.execute("CREATE VIEW elegibles AS SELECT * FROM clasificadas WHERE motivo='Conservar'")
assert remaining==con.execute('SELECT count(*) FROM elegibles').fetchone()[0]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

**02_exclusiones_por_motivo**

,motivo,contaminante,registros
0,Concentración no cuantificable,PM10,249020
1,Concentración no cuantificable,PM2.5,193740
2,Final anterior al inicio,PM10,29
3,Inicio no interpretable,PM10,196
4,Inicio no interpretable,PM2.5,210
5,Procedencia incompleta,PM10,28942
6,Procedencia incompleta,PM2.5,10242
7,Zona UTM no admitida,PM10,17677
8,Zona UTM no admitida,PM2.5,17677


## 5. Duplicados y claves ambiguas

La identidad de una observación conserva informe, evaluación, punto, coordenadas, zona, datum, inicio, periodo y final declarado. Se distingue la concentración medida para identificar duplicados exactos. Las representaciones originales no se utilizan para impedir la eliminación de repeticiones equivalentes después de normalizar.

Si varias mediciones del mismo contaminante continúan compartiendo una clave, se separan todas para revisión. El procedimiento no selecciona una concentración arbitraria ni calcula un promedio para resolver el conflicto. No exige que ambos contaminantes estén presentes: esa comprobación corresponde a la preparación de vectores.

In [6]:
OBS_KEYS=['informe','evaluacion','punto','este','norte','zona','datum','fecha','hora','periodo','fecha_fin_clave','hora_fin_clave']
DUP_KEYS=OBS_KEYS+['contaminante','concentracion','unidad','altitud']
identity=','.join(DUP_KEYS)
order=','.join(x+' ASC NULLS FIRST' for x in ['periodo_original','datum_original','fecha_original','hora_original',
 'fecha_fin_original','hora_fin_original','parametro_original','unidad_original','valor_original','altitud_original'])
con.execute('CREATE TABLE ordenadas AS SELECT *,row_number() OVER(PARTITION BY '+identity+' ORDER BY '+order+') AS replica FROM elegibles')
con.execute('CREATE VIEW unicas AS SELECT * FROM ordenadas WHERE replica=1')
n_eligible=int(con.execute('SELECT count(*) FROM elegibles').fetchone()[0])
n_unique=int(con.execute('SELECT count(*) FROM unicas').fetchone()[0])
rows.append({'etapa':'Duplicados exactos eliminados','filas':n_unique,'descartes_en_etapa':n_eligible-n_unique})
con.execute('CREATE VIEW conteos_clave AS SELECT '+','.join(OBS_KEYS)+',contaminante,count(*) AS n FROM unicas GROUP BY ALL')
con.execute('CREATE TABLE marcadas AS SELECT *,count(*) OVER(PARTITION BY '+','.join(OBS_KEYS+['contaminante'])+') AS cantidad_en_clave FROM unicas')
n_ambiguous=int(con.execute('SELECT count(*) FROM marcadas WHERE cantidad_en_clave>1').fetchone()[0])
con.execute("""CREATE TABLE limpias AS SELECT informe,evaluacion,punto,este,norte,zona,datum,altitud,
 fecha,hora,periodo,fecha_fin,hora_fin,inicio,fin,contaminante,concentracion,unidad,
 periodo_original,datum_original,fecha_original,hora_original,fecha_fin_original,hora_fin_original,
 fecha_fin_clave,hora_fin_clave,(datum='WGS84') AS datum_reconocido
 FROM marcadas WHERE cantidad_en_clave=1""")
n_clean=int(con.execute('SELECT count(*) FROM limpias').fetchone()[0])
rows.append({'etapa':'Claves ambiguas separadas','filas':n_clean,'descartes_en_etapa':n_ambiguous})
balance=pd.DataFrame(rows)
assert int(balance.descartes_en_etapa.sum())+n_clean==n_source
assert n_clean+n_ambiguous==n_unique
tables['03_balance_limpieza']=balance
balance.to_csv(OUT/'03_balance_limpieza.csv',index=False)
display(Markdown('**03_balance_limpieza**'));display(balance)
report('04_resumen_claves_ambiguas',"SELECT contaminante,count(*) AS claves,sum(n) AS mediciones FROM conteos_clave WHERE n>1 GROUP BY contaminante ORDER BY contaminante")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

**03_balance_limpieza**

,etapa,filas,descartes_en_etapa
0,Fuente completa,11407394,0
1,PM10 y PM2.5,1580630,9826764
2,Concentración no cuantificable,1137870,442760
3,Concentración negativa,1137870,0
4,Unidad no compatible,1137870,0
5,Procedencia incompleta,1098686,39184
6,Punto no informado,1098686,0
7,Inicio no interpretable,1098280,406
8,Periodo no reconocido,1098280,0
9,Zona UTM no admitida,1062926,35354


**04_resumen_claves_ambiguas**

,contaminante,claves,mediciones


,contaminante,claves,mediciones


## 6. Comprobaciones del resultado

Se verifican concentraciones finitas no negativas, unidades, identidad, inicio, periodo, referencia espacial informada y unicidad por clave. Se reportan además los finales desconocidos y los datum pendientes de revisión. Estas limitaciones no se ocultan mediante rellenos automáticos.

Las distribuciones describen mediciones conservadas. Los promedios se calculan por contaminante y periodo; no se mezclan lecturas horarias y de 24 horas para producir un promedio diario.

In [7]:
invalid=int(con.execute("""SELECT count(*) FROM limpias WHERE concentracion IS NULL OR NOT isfinite(concentracion)
 OR concentracion<0 OR unidad<>'µg/m³' OR fecha IS NULL OR hora IS NULL OR periodo NOT IN ('1 hora','24 horas')
 OR zona NOT IN (17,18,19) OR este<100000 OR este>900000 OR norte<=0 OR norte>10000000
 OR (fin IS NOT NULL AND fin<inicio)""").fetchone()[0])
assert invalid==0,'Persisten registros que incumplen los criterios.'
repeat_keys=int(con.execute('SELECT count(*) FROM (SELECT '+','.join(OBS_KEYS)+',contaminante,count(*) AS n FROM limpias GROUP BY ALL HAVING n>1)').fetchone()[0])
assert repeat_keys==0,'Persisten claves ambiguas.'
report('05_composicion_final',"""SELECT contaminante,periodo,count(*) AS mediciones,
 count(DISTINCT (punto,este,norte,zona,datum)) AS identidades_espaciales,
 min(fecha) AS primera_fecha,max(fecha) AS ultima_fecha,
 count(*) FILTER(WHERE NOT datum_reconocido) AS datum_pendiente_revision,
 count(*) FILTER(WHERE fin IS NULL) AS final_desconocido,
 count(*) FILTER(WHERE concentracion=0) AS concentraciones_cero
 FROM limpias GROUP BY ALL ORDER BY contaminante,periodo""")
report('06_cobertura_anual',"""SELECT year(fecha) AS anio,contaminante,periodo,count(*) AS mediciones
 FROM limpias GROUP BY ALL ORDER BY anio,contaminante,periodo""")
report('07_estadisticas_concentracion',"""SELECT contaminante,periodo,count(*) AS mediciones,
 min(concentracion) AS minimo,avg(concentracion) AS promedio,median(concentracion) AS mediana,
 stddev_samp(concentracion) AS desviacion_estandar,quantile_cont(concentracion,0.25) AS q1,
 quantile_cont(concentracion,0.75) AS q3,quantile_cont(concentracion,0.99) AS p99,max(concentracion) AS maximo
 FROM limpias GROUP BY ALL ORDER BY contaminante,periodo""")
report('08_referencia_espacial',"""SELECT zona,datum,datum_reconocido,count(*) AS mediciones
 FROM limpias GROUP BY ALL ORDER BY zona,datum""")
report('09_disponibilidad_por_clave',"""WITH claves AS (SELECT """+','.join(OBS_KEYS)+""",
 count(DISTINCT contaminante) AS componentes FROM limpias GROUP BY ALL)
 SELECT componentes,count(*) AS claves FROM claves GROUP BY componentes ORDER BY componentes""")

**05_composicion_final**

,contaminante,periodo,mediciones,identidades_espaciales,primera_fecha,ultima_fecha,datum_pendiente_revision,final_desconocido,concentraciones_cero
0,PM10,1 hora,574224,44,2018-01-01,2024-12-31,52457,573986,41850
1,PM10,24 horas,24278,49,2018-01-01,2024-12-31,2187,5002,1719
2,PM2.5,1 hora,446002,39,2019-04-24,2024-12-31,52457,445764,16438
3,PM2.5,24 horas,18393,42,2019-01-25,2024-12-31,2187,911,617


**06_cobertura_anual**

,anio,contaminante,periodo,mediciones
0,2018,PM10,1 hora,20415
1,2018,PM10,24 horas,978
2,2019,PM10,1 hora,25008
3,2019,PM10,24 horas,1267
4,2019,PM2.5,1 hora,4344
5,2019,PM2.5,24 horas,271
6,2020,PM10,1 hora,28776
7,2020,PM10,24 horas,1344
8,2020,PM2.5,1 hora,3168
9,2020,PM2.5,24 horas,167


**07_estadisticas_concentracion**

,contaminante,periodo,mediciones,minimo,promedio,mediana,desviacion_estandar,q1,q3,p99,maximo
0,PM10,1 hora,574224,0.0,38.638225,21.98,55.471649,9.88,45.670000,243.7000,6024.55
1,PM10,24 horas,24278,0.0,39.228661,25.90,41.050876,13.60,49.800000,194.2000,659.50
2,PM2.5,1 hora,446002,0.0,19.803778,11.32,26.936345,6.04,20.700000,137.5498,1256.30
3,PM2.5,24 horas,18393,0.0,20.036321,12.60,23.567977,7.40,21.232877,117.7080,203.20


**08_referencia_espacial**

,zona,datum,datum_reconocido,mediciones
0,17,WGS84,True,262139
1,18,WGS84,True,443897
2,18,WSG84,False,109288
3,19,WGS84,True,247573


**09_disponibilidad_por_clave**

,componentes,claves
0,1,135609
1,2,463644


,componentes,claves
0,1,135609
1,2,463644


## 7. Exportación de mediciones y auditoría

`PM25_PM10_limpio.csv` y `PM25_PM10_limpio.parquet` son los archivos de entrada para la siguiente etapa. Conservan metadatos y representaciones originales necesarias para revisar fechas, horas y referencia espacial.

`auditoria_exclusiones.parquet` contiene las mediciones de PM descartadas con el primer motivo aplicable, duplicados y conflictos de clave. Los parámetros ajenos al estudio solo se contabilizan, evitando copiar millones de filas innecesarias. El origen completo se conserva en el CSV original.

In [8]:
clean_csv=OUT/'PM25_PM10_limpio.csv'
clean_parquet=OUT/'PM25_PM10_limpio.parquet'
audit_parquet=OUT/'auditoria_exclusiones.parquet'
con.execute("COPY limpias TO '"+sqlpath(clean_csv)+"' (HEADER,DELIMITER ',')")
con.execute("COPY limpias TO '"+sqlpath(clean_parquet)+"' (FORMAT PARQUET,COMPRESSION ZSTD)")
con.execute("""CREATE VIEW auditoria AS
 SELECT informe,evaluacion,punto,este_original,norte_original,zona_original,datum_original,
 periodo_original,fecha_original,hora_original,fecha_fin_original,hora_fin_original,
 parametro_original,unidad_original,valor_original,contaminante,concentracion,motivo
 FROM clasificadas WHERE motivo<>'Conservar'
 UNION ALL SELECT informe,evaluacion,punto,este_original,norte_original,zona_original,datum_original,
 periodo_original,fecha_original,hora_original,fecha_fin_original,hora_fin_original,
 parametro_original,unidad_original,valor_original,contaminante,concentracion,'Duplicado exacto'
 FROM ordenadas WHERE replica>1
 UNION ALL SELECT informe,evaluacion,punto,este_original,norte_original,zona_original,datum_original,
 periodo_original,fecha_original,hora_original,fecha_fin_original,hora_fin_original,
 parametro_original,unidad_original,valor_original,contaminante,concentracion,'Clave ambigua'
 FROM marcadas WHERE cantidad_en_clave>1""")
con.execute("COPY auditoria TO '"+sqlpath(audit_parquet)+"' (FORMAT PARQUET,COMPRESSION ZSTD)")
n_audit=int(con.execute('SELECT count(*) FROM auditoria').fetchone()[0])
assert n_audit+n_clean==n_pm
n_exported=int(con.execute("SELECT count(*) FROM read_parquet('"+sqlpath(clean_parquet)+"')").fetchone()[0])
assert n_exported==n_clean
manifest={'run_id':RUN_ID,'etapa':'limpieza','estado':'completado','csv':str(clean_csv),
 'parquet':str(clean_parquet),'auditoria':str(audit_parquet),'filas':n_clean,
 'componentes':['PM10','PM2.5'],'unidad':'µg/m³','periodo':'todos los años disponibles',
 'clave_observacion':OBS_KEYS,'clave_duplicados':DUP_KEYS,
 'finales_ausentes':'se conservan sin inferir','datum_no_reconocido':'conservado y señalado para revisión',
 'columnas':list(con.execute('DESCRIBE limpias').df().column_name)}
(OUT/'manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
display(pd.DataFrame([{'archivo':p.name,'bytes':p.stat().st_size} for p in [clean_csv,clean_parquet,audit_parquet]]))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,archivo,bytes
0,PM25_PM10_limpio.csv,284241083
1,PM25_PM10_limpio.parquet,10104612
2,auditoria_exclusiones.parquet,345939


## 8. Resumen del procedimiento

Los resultados se calculan al ejecutar el cuaderno. El resumen distingue las filas de la fuente, las mediciones de los contaminantes seleccionados, las exclusiones y las mediciones finales. No se presenta el conteo como número de vectores ni como evidencia de ejecución concurrente en Go.

In [9]:
first_date,last_date=con.execute('SELECT min(fecha),max(fecha) FROM limpias').fetchone()
by_pm=dict(con.execute('SELECT contaminante,count(*) FROM limpias GROUP BY contaminante').fetchall())
n_datum=int(con.execute('SELECT count(*) FROM limpias WHERE NOT datum_reconocido').fetchone()[0])
n_unknown_end=int(con.execute('SELECT count(*) FROM limpias WHERE fin IS NULL').fetchone()[0])
summary={'registros_fuente':n_source,'mediciones_PM_entrada':n_pm,'mediciones_PM_limpias':n_clean,
 'mediciones_por_contaminante':by_pm,'exclusiones_PM':n_audit,
 'duplicados_eliminados':n_eligible-n_unique,'mediciones_con_clave_ambigua':n_ambiguous,
 'datum_pendiente_revision':n_datum,'final_desconocido':n_unknown_end,
 'fecha_minima':str(first_date),'fecha_maxima':str(last_date),'filtro_de_antiguedad':None,
 'porcentaje_conservado_PM':round(100*n_clean/n_pm,2) if n_pm else None}
text=f"""# Resumen de la limpieza

La fuente contiene **{n_source:,} registros**. Se seleccionaron **{n_pm:,} mediciones de PM10 y PM2.5** y se conservaron **{n_clean:,}** después de las exclusiones, la normalización y el control de claves.

El conjunto conserva **{by_pm.get('PM10',0):,} mediciones de PM10** y **{by_pm.get('PM2.5',0):,} de PM2.5** entre **{first_date} y {last_date}**, sin filtro de antigüedad. La proporción conservada es **{summary['porcentaje_conservado_PM']} %** del subconjunto PM.

Se retiraron **{n_eligible-n_unique:,} duplicados** en la clave documentada y **{n_ambiguous:,} mediciones con clave ambigua**. Las exclusiones de PM suman **{n_audit:,} registros** y se conservan con su motivo en la auditoría. El balance satisface entrada PM = exclusiones PM + mediciones finales.

Persisten **{n_datum:,} mediciones con datum pendiente de revisión** y **{n_unknown_end:,} sin final interpretable**. No se ha inventado información para completarlas. Los extremos y los ceros se conservan. Los promedios se describen por contaminante y periodo.

El manifiesto `Limpieza_PM/ultima_ejecucion.json` identifica los archivos que utilizará la preparación de vectores. Cada fila de estos archivos sigue siendo una medición individual.
"""
display(Markdown(text))
(OUT/'RESUMEN_LIMPIEZA.md').write_text(text,encoding='utf-8')
(OUT/'resumen_limpieza.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding='utf-8')
provenance.update(estado='completado',resultados=summary)
(OUT/'ejecucion.json').write_text(json.dumps(provenance,indent=2,ensure_ascii=False),encoding='utf-8')
(DATA_DIR/'Limpieza_PM'/'ultima_ejecucion.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
con.close()

# Resumen de la limpieza

La fuente contiene **11,407,394 registros**. Se seleccionaron **1,580,630 mediciones de PM10 y PM2.5** y se conservaron **1,062,897** después de las exclusiones, la normalización y el control de claves.

El conjunto conserva **598,502 mediciones de PM10** y **464,395 de PM2.5** entre **2018-01-01 y 2024-12-31**, sin filtro de antigüedad. La proporción conservada es **67.25 %** del subconjunto PM.

Se retiraron **0 duplicados** en la clave documentada y **0 mediciones con clave ambigua**. Las exclusiones de PM suman **517,733 registros** y se conservan con su motivo en la auditoría. El balance satisface entrada PM = exclusiones PM + mediciones finales.

Persisten **109,288 mediciones con datum pendiente de revisión** y **1,025,663 sin final interpretable**. No se ha inventado información para completarlas. Los extremos y los ceros se conservan. Los promedios se describen por contaminante y periodo.

El manifiesto `Limpieza_PM/ultima_ejecucion.json` identifica los archivos que utilizará la preparación de vectores. Cada fila de estos archivos sigue siendo una medición individual.
